# Class-merge sensitivity check (final model)

**Question:** the label-overlap check showed that about 80% of Analysis and Backdoor rows have an exact feature twin with another label. How much of the macro F1 gap is caused by those two classes, and does reducing the classes help the GNN more than the tree model?

**Method:** no retraining. The test predictions of the final model (two-stage LightGBM trained on the original data with class weights, 3 seeds, saved by the TTL-ablation notebook) and the saved GNN test probabilities are relabelled after the fact and macro F1 is recomputed. These rows change the task, so they are a *sensitivity analysis*, not results to compare with other 10-class work.

In [ ]:
import glob

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

_fig_count = 0
_plt_show = plt.show


def show_and_save(*args, **kwargs):
    """Show every figure and also keep it as a PNG in the notebook output."""
    global _fig_count
    _fig_count += 1
    plt.savefig(f"/kaggle/working/fig_{_fig_count:02d}.png", dpi=130, bbox_inches="tight")
    _plt_show(*args, **kwargs)


plt.show = show_and_save

DATASET_PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
find = lambda name: glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]

y = pd.read_parquet(f"{DATASET_PATH}/test.parquet", columns=["label"])
class_names = list(joblib.load(f"{DATASET_PATH}/preprocessing.pkl")["label_encoder"].classes_)
tab_preds = pd.read_parquet(find("final_model_test_pred.parquet")).loc[y.index]     # aligned by row_id, one column per seed
gnn_pred = np.load(find("gnn_test_proba.npy")).argmax(1)                              # same row order as test.parquet
y = y["label"].to_numpy()
ids = list(range(len(class_names)))
print(class_names, "| test rows:", len(y), "| tabular seeds:", list(tab_preds.columns))

## Per-class F1 of the final model (mean of 3 seeds)

In [ ]:
f1 = pd.DataFrame({c: f1_score(y, tab_preds[c].to_numpy(), labels=ids, average=None, zero_division=0) for c in tab_preds}, index=class_names)
f1_mean = f1.mean(axis=1)
print(pd.DataFrame({"mean": f1_mean, "std": f1.std(axis=1)}).round(3).to_string())
f1_mean.sort_values().plot.barh(figsize=(7, 4), color="steelblue")
plt.xlabel("test F1")
plt.title("Per-class F1, final two-stage LightGBM (macro F1 = %.3f)" % f1_mean.mean())
plt.tight_layout()
plt.show()

## Relabel and recompute

Analysis and Backdoor collide mostly with large classes (DoS, Exploits, Fuzzers, Generic, Reconnaissance), so besides merging the two we also fold them into Exploits and remove their rows from scoring. Each view is applied identically to the tree model (mean and std over its 3 seeds) and to the GNN (single saved run).

In [ ]:
A, B, X = (class_names.index(c) for c in ["Analysis", "Backdoor", "Exploits"])
without_ab = [i for i in ids if i not in (A, B)]
keep = ~np.isin(y, [A, B])


def macro(yt, yp, labels):
    return f1_score(yt, yp, labels=labels, average="macro", zero_division=0)


def class_views(p):
    return {
        "10 classes (original)": macro(y, p, ids),
        "Analysis + Backdoor merged (9)": macro(np.where(y == B, A, y), np.where(p == B, A, p), [i for i in ids if i != B]),
        "Analysis + Backdoor folded into Exploits (8)": macro(np.where(np.isin(y, [A, B]), X, y), np.where(np.isin(p, [A, B]), X, p), without_ab),
        "Analysis + Backdoor removed (8)": macro(y[keep], p[keep], without_ab),
    }


tab_views = pd.DataFrame({c: class_views(tab_preds[c].to_numpy()) for c in tab_preds})
results = pd.DataFrame({"tabular mean": tab_views.mean(axis=1), "tabular std": tab_views.std(axis=1), "GNN": pd.Series(class_views(gnn_pred))})
results["gap (tabular - GNN)"] = results["tabular mean"] - results["GNN"]
print(results.round(3).to_string())

results[["tabular mean", "GNN"]].iloc[::-1].plot.barh(figsize=(10, 4.5), color=["darkorange", "steelblue"])
plt.xlabel("test macro F1")
plt.title("Same predictions, different class definitions")
plt.tight_layout()
plt.show()